# safety_report

> Summaries and the auto-written report for Phase 9: attacks, defences (cost and benefit), monitoring, sanctions, liar detection, the free-riding index and H3.

Spec: `specs/multiagent/phase9/` (design-low-level section 6).

In [ ]:
#| default_exp safety_report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from explRL.bench_report import _f, _mean
from explRL.bench_stats import bootCI
from explRL.safety_runs import lossRows


def group(rows, keys, metrics):
    "Rows grouped by `keys`: per group and metric (mean, CI low, CI high), plus the row count."
    g = {}
    for r in rows:
        g.setdefault(tuple(r[k] for k in keys), []).append(r)
    return {k: {**{m: (_mean([float(r[m]) for r in v]), *bootCI([float(r[m]) for r in v])) for m in metrics}, 'n': len(v)}
            for k, v in sorted(g.items(), key=lambda kv: tuple((0, float(x), '') if isinstance(x, (int, float))
                                                          else (1, 0.0, str(x)) for x in kv[0]))}


def _by(rows, **kw):
    return [r for r in rows if all(r.get(k) == v for k, v in kw.items())]


def _top(N):
    return max(1, round(0.25 * int(N)))


def costBenefit(rows):
    "Per channel, defence and N: cost without attack (1 - Wh / undefended clean Wh) and benefit under 25% liars."
    att = _by(rows, study='attack')
    clean = {(r['channel'], int(r['N']), int(r['seed'])): r['Wh'] for r in att if int(r['nBad']) == 0}
    liars = {(r['channel'], int(r['N']), int(r['seed'])): r['Wh'] for r in att
             if r['kind'] == 'liar' and int(r['nBad']) == _top(r['N'])}
    out = {}
    for r in _by(rows, study='defence'):
        k = (r['channel'], int(r['N']), int(r['seed']))
        d = out.setdefault((r['channel'], r['defence'], int(r['N'])), dict(cost=[], benefit=[]))
        if int(r['nBad']) == 0 and k in clean:
            d['cost'].append(1 - r['Wh'] / clean[k])
        elif r['kind'] == 'liar' and int(r['nBad']) == _top(r['N']) and k in liars and k in clean:
            d['benefit'].append((r['Wh'] - liars[k]) / clean[k])
    return {k: {m: (_mean(v[m]), *bootCI(v[m])) for m in v} for k, v in sorted(out.items())}


def summaries(rows):
    "All exploratory tables of the Phase 9 report."
    loss = lossRows(rows)
    return dict(
        attack=group(_by(rows, study='attack'), ('channel', 'N', 'kind', 'nBad'), ('Wh', 'Wt')),
        attackLoss=group(_by(loss, study='attack'), ('channel', 'N', 'kind', 'nBad'), ('loss', 'lossTeam')),
        defence=group(_by(rows, study='defence'), ('channel', 'defence', 'N', 'kind', 'nBad'), ('Wh', 'Wt')),
        costBenefit=costBenefit(rows),
        monitor=group(_by(rows, study='monitor'), ('defence', 'q', 'N', 'nBad'),
                      ('Wh', 'monitorCost', 'honestLevel2', 'badRevoked')),
        sanctions=group([r for r in rows if r['defence'] == 'sanctions'], ('study', 'channel', 'N', 'kind', 'nBad'),
                        ('badRevoked', 'honestLevel2', 'firstRevoke')),
        detection=group([r for r in rows if r['kind'] == 'liar' and r['channel'] in ('blackboard', 'messages')],
                        ('channel', 'defence', 'N'), ('aucWhite', 'aucBlack')),
        freeRiding=group([r for r in _by(rows, study='attack') if r['kind'] in ('none', 'freerider')],
                         ('channel', 'N', 'kind', 'nBad'), ('freeRiding',)),
        h3=group(_by(loss, study='h3'), ('channel', 'N'), ('loss', 'lossTeam')),
        h3Clean=group([r for r in _by(rows, study='h3') if int(r['nBad']) == 0], ('channel', 'N'), ('Wh',)))


FIGURES = dict(
    attack='honest welfare per honest agent-second vs number of bad agents, per kind and size, one line per channel',
    team='the same for team welfare (all agents)',
    defence='honest welfare vs number of bad agents per defence (blackboard and messages)',
    cost='each defence: cost without attack and benefit under 25% liars, relative to undefended clean welfare',
    monitor='honest welfare (clean and 25% liars) vs monitoring cost, one point per spot-check rate q, per defence',
    detection='detection AUC from beliefs vs reports (white-box) and from the defences (black-box)',
    h3='H3: relative honest-welfare loss from one liar per size, blackboard and messages')


def _table(title, tab, cols, fmt=_f):
    if not tab:
        return []
    L = [f"**{title}**\n", "| group | n | " + ' | '.join(cols) + " |", "|---|---|" + '---|' * len(cols)]
    for k, v in tab.items():
        L.append(f"| {' / '.join(str(int(x)) if isinstance(x, float) and x == int(x) else str(x) for x in k)} | {v['n']} | "
                 + ' | '.join(f"{fmt(v[c][0])} [{fmt(v[c][1])}, {fmt(v[c][2])}]" for c in cols) + " |")
    return L + [""]


def report(meta, conf=None, tabs=None):
    "The Phase 9 report.md text; everything but the H3 table is exploratory."
    L = [f"# Phase 9 safety benchmark ({meta['mode']} run)\n",
         f"{meta['cells']} cells, {meta['workers']} worker processes, {meta['wallSeconds']:.0f} s. Seeds: {meta['seeds']}. "
         f"H3 pre-registration hash `{meta['preregHash']}`, git commit `{meta['gitCommit']}`. "
         f"Deviations from the pre-registration: {meta.get('deviations', 'none')}. Strike threshold theta = {meta['theta']}.\n"]
    if conf:
        c = conf['H3']
        L += ["## H3 (confirmatory)\n"] + (["*Quick run on pilot seeds with scaled budgets: a pipeline check, not a result.*\n"]
                                          if meta['mode'] == 'quick' else [])
        L += ["| test | contrast | mean [95% CI] | dz | p | verdict |", "|---|---|---|---|---|---|",
              f"| H3 | relative honest-welfare loss from one liar, blackboard - messages, sizes 8-32 | {_f(c['mean'])} "
              f"[{_f(c['ci'][0])}, {_f(c['ci'][1])}] | {_f(c['dz'], 2)} | {_f(c['p'], 4)} | **{c['verdict']}** |", ""]
    tabs = tabs or {}
    L += _table('H3 cells: relative loss per channel and size', tabs.get('h3'), ('loss', 'lossTeam'))
    L += _table('H3 cells: clean honest welfare per channel and size', tabs.get('h3Clean'), ('Wh',))
    if tabs.get('attack'):
        L.append("## Exploratory\n\nGroups are channel / size / kind / number of bad agents unless stated; values are seed means [95% CI].\n")
    L += _table('Attacks: honest and team welfare (no defence)', tabs.get('attack'), ('Wh', 'Wt'))
    L += _table('Attacks: relative loss vs clean (honest, team)', tabs.get('attackLoss'), ('loss', 'lossTeam'))
    L += _table('Defences: honest and team welfare (channel / defence / size / kind / count)', tabs.get('defence'), ('Wh', 'Wt'))
    cb = {k: {**v, 'n': ''} for k, v in (tabs.get('costBenefit') or {}).items()}
    L += _table('Defence cost without attack and benefit under 25% liars (channel / defence / size)', cb, ('cost', 'benefit'))
    L += _table('Monitoring sweep, blackboard (defence / q / size / liars)', tabs.get('monitor'),
                ('Wh', 'monitorCost', 'honestLevel2', 'badRevoked'))
    L += _table('Sanctions (grid / channel / size / kind / count)', tabs.get('sanctions'), ('badRevoked', 'honestLevel2', 'firstRevoke'))
    L += _table('Liar detection AUC (channel / defence / size)', tabs.get('detection'), ('aucWhite', 'aucBlack'))
    L += _table('Free-riding index: clean vs free-rider teams', tabs.get('freeRiding'), ('freeRiding',))
    if meta.get('figures'):
        L += ["## Figures\n"] + [f"- `{n}.png`: {FIGURES[n]}" for n in meta['figures']] + [""]
    L.append("Re-run: `nbdev-test --n-workers 0 nbs/demo_safety.ipynb` (quick), `--cell-timeout 0 --flags slow` (H3), "
             "`--cell-timeout 0 --flags sweeps` (exploratory grids). Spec: `specs/multiagent/phase9/`.\n")
    return '\n'.join(L)

## Tests

In [ ]:
import numpy as np
from fastcore.test import test_eq, test_close
from explRL.bench_grid import runGrid, confirm
from explRL.safety_grid import PREREG_H3, preregHashH3, attackGrid, defenceGrid, monitorGrid, h3Grid, runSafetyCell
from explRL.safety_report import group, costBenefit, summaries, report
from explRL.safety_runs import lossRows

g = group([dict(a=1, x=1.0), dict(a=1, x=3.0), dict(a=2, x=5.0)], ('a',), ('x',))
test_eq(list(g), [(1,), (2,)]); test_close(g[(1,)]['x'][0], 2.0); test_eq(g[(1,)]['n'], 2)
test_eq(list(group([dict(N=16.0, y=1.0), dict(N=8.0, y=1.0)], ('N',), ('y',))), [(8.0,), (16.0,)])   # numeric order
rows = runGrid(attackGrid('quick') + defenceGrid('quick') + monitorGrid('quick') + h3Grid('quick'), 2, run=runSafetyCell)
tabs = summaries(rows)
test_eq(set(tabs), {'attack', 'attackLoss', 'defence', 'costBenefit', 'monitor', 'sanctions', 'detection', 'freeRiding',
                    'h3', 'h3Clean'})
assert ('blackboard', 'trust', 8) in tabs['costBenefit'] and len(tabs['h3']) == 2
conf = confirm(lossRows(rows), PREREG_H3, seeds=(1000, 1001))
meta = dict(mode='quick', cells=len(rows), workers=2, wallSeconds=1.0, seeds='pilot 1000-1001', preregHash=preregHashH3(),
            gitCommit='abc', theta=2.5, figures=['attack', 'h3'])
txt = report(meta, conf, tabs)
for s in ('## H3 (confirmatory)', 'pipeline check', 'H3 cells', '## Exploratory', 'Defence cost', 'Monitoring sweep',
          'Liar detection', 'Free-riding', '## Figures', 'theta = 2.5'):
    assert s in txt, s

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()